# azroute: one-shot test of a trained network

Routes every case **from scratch** with a trained `model.pt`, with no training and no polish, so the score
shows what the network itself does:

* each case starts from its initial routing (negotiated congestion), then gets `MINUTES_PER_CASE` of
  greedy rerouting plus search-guided reroutes; a reroute is kept only if it does not increase the delay;
* **network** arm: the network guides the search. **control** arm: the same visits, network-free.
  Both run at the same time on half the workers each, so the difference is what the network adds;
* each arm ends with a checked export (`.sol.json`, `runtime.json`, `summary.md`) on Drive.

**Use:** Runtime > Change runtime type > **G4** GPU. Fill in the settings, run the cells top to bottom,
re-run cell 7 to watch progress. Needs `evaluate.py` in the repo that `REPO_URL` points at.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/Dev077/kappa.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
CHALLENGE_URL = "https://github.com/partcleda/eda-3d-routing-challenge.git"  #@param {type:"string"}
MODEL_PATH = "/content/drive/MyDrive/azroute_run/model.pt"  #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/azroute_eval"  #@param {type:"string"}
MINUTES_PER_CASE = 10  #@param {type:"number"}
ARMS = "network,control"  #@param {type:"string"}
TIERS = "all"  #@param {type:"string"}
WORKERS = 0  #@param {type:"integer"}
#@markdown `MODEL_PATH`: the training run's `model.pt` (synced to its `DRIVE_DIR`).
#@markdown `ARMS`: `network,control` to compare, or just `network`. `WORKERS`: total over both arms, `0` = auto.
#@markdown `TIERS`: `all` or a comma list of intro,hard,scale,stress,congested,designs.
import time
CODE_DIR = '/content/kappa'
RUN_TAG = time.strftime('%Y%m%d_%H%M')   # every test gets fresh work and output folders

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 3. Get the code, build the router core, run the tests
import os, subprocess
os.chdir('/content')
def sh(cmd, cwd='/content'):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode: raise RuntimeError(f'failed: {cmd}')

if not os.path.isdir('/content/eda-3d-routing-challenge'):
    sh(f'git clone --depth 1 {CHALLENGE_URL} eda-3d-routing-challenge')
else:
    sh('git -C eda-3d-routing-challenge pull --ff-only')
if not os.path.isdir(CODE_DIR):
    sh(f'git clone -b {BRANCH} {REPO_URL} {CODE_DIR}')
else:
    sh(f'git -C {CODE_DIR} pull --ff-only')
assert os.path.isfile(os.path.join(CODE_DIR, 'pipeline', 'evaluate.py')), 'evaluate.py is missing: push it to the repo'
os.environ['AZROUTE_CHALLENGE_ROOT'] = '/content/eda-3d-routing-challenge'
os.environ['PYTHONPATH'] = CODE_DIR
sh('pip -q install pybind11 psutil')
sh('python setup.py > build.log 2>&1 || (tail -40 build.log; exit 1)', cwd=CODE_DIR)
sh('python -m unittest tests.test_core', cwd=CODE_DIR)
print('ready:', CODE_DIR)

In [ ]:
#@title 4. Hardware and model
import sys, torch, psutil
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('cpus:', os.cpu_count(), '| RAM GB:', round(psutil.virtual_memory().total / 2**30, 1))
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > G4'
assert os.path.exists(MODEL_PATH), f'no model at {MODEL_PATH}'
ck = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
print('model:', MODEL_PATH, '|', ck.get('steps', '?'), 'training steps | net_ready:', ck.get('net_ready'))

In [ ]:
#@title 5. Start the test in the background (one process per arm)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
from pipeline.run import auto_workers
arms = [x.strip() for x in ARMS.split(',') if x.strip()]
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
per_arm = max(1, auto_workers(WORKERS, gpu_gb) // len(arms))
WORK = {arm: f'/content/eval_{arm}_{RUN_TAG}' for arm in arms}
OUT = {arm: os.path.join(OUT_DIR, RUN_TAG, arm) for arm in arms}
PROCS = {}
for arm in arms:
    os.makedirs(WORK[arm], exist_ok=True)
    args = ['python', '-m', 'pipeline.evaluate', '--arm', arm, '--work', WORK[arm], '--out', OUT[arm],
            '--minutes', str(MINUTES_PER_CASE), '--tiers', TIERS, '--workers', str(per_arm)]
    if arm == 'network':
        args += ['--model', MODEL_PATH]
    logf = open(os.path.join(WORK[arm], 'eval.log'), 'a')
    PROCS[arm] = subprocess.Popen(args, cwd=CODE_DIR, env=dict(os.environ), stdout=logf,
                                  stderr=subprocess.STDOUT, start_new_session=True)
    print(arm, 'started pid', PROCS[arm].pid, '|', per_arm, 'workers | work', WORK[arm], '| export', OUT[arm])

Each arm stops once every case has used its minutes, stalled for 8 visits, or come within 0.3 % of its
lower bound. With ~22 workers per arm and 10 minutes per case, expect about 30-45 minutes.

In [ ]:
#@title 6. Stop early (optional): workers finish their current reroute, then each arm exports
for arm in WORK:
    open(os.path.join(WORK[arm], 'STOP'), 'w').close()
print('stop requested; watch cell 7')

In [ ]:
#@title 7. Monitor (re-run any time): score per tier, network vs control
import json, math, collections
TIER_ORDER = ['intro', 'hard', 'scale', 'stress', 'congested', 'designs']

def load_stats(arm):
    p = os.path.join(WORK[arm], 'case_stats.json')
    return {k: v for k, v in json.load(open(p)).items() if not k.startswith('_')} if os.path.exists(p) else {}

def tier_scores(stats):
    by = collections.defaultdict(list)
    for s in stats.values():
        if s.get('best'):
            by[s['tier']].append(s)
    return {t: (math.exp(sum(math.log(s['baseline'] / s['best']) for s in rs) / len(rs)), len(rs),
                sum(s['seconds'] for s in rs) / 60) for t, rs in by.items()}

for arm in WORK:
    state = 'running' if PROCS[arm].poll() is None else f'finished (exit {PROCS[arm].returncode})'
    print(f'--- {arm}: {state}')
    print(''.join(open(os.path.join(WORK[arm], 'eval.log')).readlines()[-4:]))
S = {arm: tier_scores(load_stats(arm)) for arm in WORK}
hdr = f"{'tier':10}" + ''.join(f"{arm + ' score':>16}{'cases':>6}{'cpu min':>9}" for arm in WORK)
if len(WORK) == 2:
    hdr += f"{'network/control':>17}"
print(hdr)
for t in TIER_ORDER:
    if not any(t in S[arm] for arm in WORK):
        continue
    row = f'{t:10}'
    for arm in WORK:
        g, n, m = S[arm].get(t, (float('nan'), 0, 0.0))
        row += f'{g:16.4f}{n:6d}{m:9.1f}'
    if len(WORK) == 2 and all(t in S[arm] for arm in ('network', 'control')):
        row += f"{S['network'][t][0] / S['control'][t][0]:17.4f}"
    print(row)

In [ ]:
#@title 8. Results (after both arms finish): per case, network vs control
assert all(p.poll() is not None for p in PROCS.values()), 'still running: wait, or stop early with cell 6'
st = {arm: load_stats(arm) for arm in WORK}
arms_ = list(WORK)
print(f"{'case':22}{'baseline':>10}{'initial':>10}" + ''.join(f'{a:>10}' for a in arms_)
      + ''.join(f'{a + " /LB":>14}' for a in arms_))
for t in TIER_ORDER:
    for k in sorted(k for k in st[arms_[0]] if k.startswith(t + '/')):
        s0 = st[arms_[0]][k]
        row = f"{k:22}{s0['baseline']:10d}{s0.get('initial_delay', 0):10d}"
        row += ''.join(f"{st[a].get(k, {}).get('best', 0):10d}" for a in arms_)
        row += ''.join(f"{st[a].get(k, {}).get('best', 0) / max(1, st[a].get(k, {}).get('lower_bound', 1)):14.4f}" for a in arms_)
        print(row)
if len(arms_) == 2:
    wins = sum(st['network'][k].get('best', 1e18) < st['control'].get(k, {}).get('best', 1e18) for k in st['network'])
    ties = sum(st['network'][k].get('best') == st['control'].get(k, {}).get('best') for k in st['network'])
    print(f'\nnetwork better on {wins} cases, equal on {ties}, worse on {len(st["network"]) - wins - ties}')
for arm in WORK:   # keep the logs with the export
    for f in ('eval.log', 'case_stats.json', 'visits.jsonl', 'eval_config.json'):
        if os.path.exists(os.path.join(WORK[arm], f)):
            sh(f'cp {os.path.join(WORK[arm], f)} {OUT[arm]}/')
    print(arm, 'export + logs in', OUT[arm])